# CUDA graph + compile benchmark

Run the fixed-slot differentiable InEKF with batch 7, 300-row chunks, 10 chunks, five repetitions, float64, and forward + backward. Set `LEG_BICAL_DATA_ROOT`; the median and raw samples are written under the ignored `runs/notebook_benchmark` directory.

Reference runs on an RTX 5090 Laptop GPU with Torch 2.12 and CUDA 13 measured about **0.8–0.9 ms/step**, **8k batched rows/s**, and **0.17 GB** peak memory. Treat this as a machine-specific runtime record, not an estimation-quality claim.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import torch


def find_root(start):
    for path in [start, *start.parents]:
        if (path / "benchmarks" / "profile_replay.py").exists():
            return path
    raise FileNotFoundError("run this notebook from the cuda checkout")


ROOT = find_root(Path.cwd())
DATA_ROOT = Path(os.environ["LEG_BICAL_DATA_ROOT"]).expanduser()
OUT = ROOT / "runs" / "notebook_benchmark"
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this notebook")
print(torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

In [ ]:
command = [
    sys.executable,
    str(ROOT / "benchmarks" / "profile_replay.py"),
    "--impl", "fixed",
    "--batch", "7",
    "--rows", "300",
    "--chunks", "10",
    "--repeat", "5",
    "--with-grad",
    "--compile", "cuda-graph-compile",
    "--data-root", str(DATA_ROOT),
    "--out", str(OUT),
]
environment = os.environ.copy()
environment["PYTHONPATH"] = str(ROOT / "src")
run = subprocess.run(
    command, cwd=ROOT, env=environment, text=True, capture_output=True
)
if run.returncode:
    print(run.stdout)
    print(run.stderr, file=sys.stderr)
    run.check_returncode()

summary_path = OUT / "fixed_b7_ccuda-graph-compile_float64_grad.summary.json"
summary = json.loads(summary_path.read_text())
measurement = summary["measurements"]
record = {
    "gpu": summary["environment"]["gpu"],
    "torch": summary["environment"]["torch"],
    "cuda": summary["environment"]["cuda_runtime"],
    "commit": summary["commit"],
    "ms_per_step": measurement["ms_per_step"],
    "ms_per_step_samples": measurement["ms_per_step_samples"],
    "rows_per_s": measurement["rows_per_s"],
    "peak_gb": measurement["peak_gb"],
}
record